In [17]:

import os
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

DATA_DAILY = Path("data/daily")
DATA_AVERAGE = Path("data/averages")
OUTPUT_AVERAGES = Path("output/averages")

for p in (DATA_AVERAGE, DATA_DAILY, OUTPUT_AVERAGES):
    p.mkdir(parents=True, exist_ok=True)

In [18]:

# Load all hot_stocks CSVs
files = sorted([f for f in DATA_DAILY.glob("hot_stocks_*.csv")])
if not files:
    raise ValueError("⚠ No daily hot stocks files found in data/daily/. Cannot compute averages.")

all_days = []
for f in files:
    date = f.stem.split("_")[-1]  # get YYYYMMDDhhmmss
    df = pd.read_csv(f)
    df["date"] = pd.to_datetime(date, format="%Y%m%d%H%M%S")
    all_days.append(df)

full_df = pd.concat(all_days, ignore_index=True)

full_df.head(4)


,symbol,regularMarketPrice,regularMarketChangePercent,regularMarketVolume,averageDailyVolume3Month,marketCap,VolumeSpike,MomentumScore,VolumeScore,VolatilityScore,TrendScore,HotScore,date
0,DAVE,306.355,6.825794,1099029.0,574007.0,3.895073e+09,1.914661,0.813397,0.964115,0.933014,0.959330,0.904665,2026-06-16 10:45:16
1,MRVL,311.290,11.294242,44084905.0,34943947.0,2.725510e+11,1.261589,0.961722,0.885167,0.976077,0.550239,0.896651,2026-06-16 10:45:16
2,ENTG,163.020,8.304544,3682585.0,2792768.0,2.482795e+10,1.318615,0.901914,0.897129,0.875598,0.825359,0.887321,2026-06-16 10:45:16
3,OMAB,108.940,7.045304,170231.0,84719.0,5.258662e+09,2.009360,0.830144,0.968900,0.796651,0.978469,0.886842,2026-06-16 10:45:16


In [19]:
# Metrics to average
metrics = [
    "HotScore",
    "regularMarketPrice",
    "regularMarketChangePercent",
    "VolumeSpike",
    "MomentumScore",
    "VolumeScore",
    "VolatilityScore",
    "TrendScore"
]
metrics = [m for m in metrics if m in full_df.columns]

# Compute averages
avg_df = full_df.groupby("symbol")[metrics].mean().reset_index().sort_values("HotScore", ascending=False)
avg_csv = DATA_AVERAGE / "average_hot_scores.csv"
avg_df.to_csv(avg_csv) 

In [20]:
avg_df.head()

,symbol,HotScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,MomentumScore,VolumeScore,VolatilityScore,TrendScore
138,NTRA,0.951486,259.970000,10.738600,1.983736,0.980892,0.927813,0.974522,0.885350
106,KBH,0.947771,61.510000,16.650900,3.637814,1.000000,0.991507,0.887473,0.732484
153,QURE,0.947298,48.453333,55.782737,8.019678,0.997475,0.964646,0.915725,0.774102
96,ICLR,0.946603,158.170000,10.864200,2.053477,0.983015,0.929936,0.954352,0.861996
122,MANE,0.942038,117.290000,13.214300,1.764692,0.993631,0.900212,0.946921,0.898089


### colors 
aggrnyl     agsunset    blackbody   bluered     blues       blugrn      bluyl       brwnyl
bugn        bupu        burg        burgyl      cividis     darkmint    electric    emrld
gnbu        greens      greys       hot         inferno     jet         magenta     magma
mint        orrd        oranges     oryel       peach       pinkyl      plasma      plotly3
pubu        pubugn      purd        purp        purples     purpor      rainbow     rdbu
rdpu        redor       reds        sunset      sunsetdark  teal        tealgrn     turbo
viridis     ylgn        ylgnbu      ylorbr      ylorrd      algae       amp         deep
dense       gray        haline      ice         matter      solar       speed       tempo
thermal     turbid      armyrose    brbg        earth       fall        geyser      prgn
piyg        picnic      portland    puor        rdgy        rdylbu      rdylgn      spectral
tealrose    temps       tropic      balance     curl        delta       oxy         edge
hsv         icefire     phase       twilight    mrybm       mygbm

In [21]:
import plotly.express as px

avg_hot_score_csv = avg_df.sort_values(by=["HotScore"], ascending=True).head(40)

fig = px.bar(   
    avg_hot_score_csv, 
    x="HotScore",
    y="symbol", 
    orientation="h",
    color="VolatilityScore",
    text="regularMarketPrice",
    hover_data=["regularMarketPrice","TrendScore"], 
    title="Average Hot Score vs Average Volume Spike",
    color_continuous_scale="ice",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 40 Average Hot Score Stocks",
    xaxis_title="Hot Score",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1200,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()
    
chart_path = os.path.join(OUTPUT_AVERAGES, "bar_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [22]:
import plotly.express as px

avg_hot_score_csv = avg_df.sort_values(by=["regularMarketPrice", "regularMarketChangePercent"], ascending=True).head(40)

fig = px.bar(   
    avg_hot_score_csv, 
    x="regularMarketPrice",
    y="symbol", 
    orientation="h",
    color="HotScore",
    text="regularMarketPrice",
    hover_data=["TrendScore","VolatilityScore"], 
    title="Average Hot Score vs Average Volume Spike",
    color_continuous_scale="Blues",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 40 Average Regular Market Price Stocks",
    xaxis_title="regular Market Price",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1200,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()
    
chart_path = os.path.join(OUTPUT_AVERAGES, "regular_bar_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [23]:
avg_volatility = avg_df.sort_values("VolatilityScore", ascending=True).head(40)


fig = px.scatter(
    avg_volatility,
    x="VolatilityScore",
    y="HotScore",
    size="TrendScore",
    color="MomentumScore",
    text="regularMarketPrice",
    #hover_data=["regularMarketPrice"], 
    hover_name="symbol",
    title="Top 40 Average Hot Score Stocks by Volatility and Momentum",
    color_continuous_scale="ice",
)

fig.update_traces(textposition="top right", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 40 Average Hot Score Stocks by Volatility and Momentum", 
    xaxis_title="Volatility Score",
    yaxis_title="Hot Score",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

chart_path = os.path.join(OUTPUT_AVERAGES, "scatter_volatility_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [24]:

avg_trend_score = avg_df.sort_values("TrendScore", ascending=True).head(40)

fig = px.bar_polar(
    avg_trend_score,
    r="TrendScore",
    theta="symbol",
    color="VolatilityScore",
    hover_name="symbol",
    hover_data=["regularMarketPrice", "VolatilityScore"], 
    title="Top 40 Average Trend Score Stocks by Volatility",
    color_continuous_scale="ice",
)
  

fig.update_layout(
    title="Top 40 Average Trend Score Stocks by Volatility", 
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

chart_paths = os.path.join(OUTPUT_AVERAGES, "bar_polar_trend_score.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")